# Credential theft prevention and lateral movement  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 5.2.c: Detect lateral movement in an authentication log

**Skill:** Mitigate Risk, Detect Attacks · **Maps to:** 5.2.B, 5.2.C · **Language:** Python 3 (pytest)

## Problem
Build the detector a security team's SIEM rule would run over authentication events. Given a list
of authentication events (account, destination host, timestamp in minutes), flag any account that
authenticated to **at least `threshold` distinct hosts** within any rolling **`window_minutes`**
window, a classic lateral-movement signal.

## Function contract
Implement `solve(events: list, threshold: int, window_minutes: int) -> list` in the code editor
(see the Reference section below).

## Input
- `events`: `list[dict]`, each `{"account": str, "host": str, "ts": int}` (`ts` = minutes since
  start of log, non-negative integer, not necessarily sorted).
- `threshold`: `int`, minimum number of **distinct** hosts within the window to flag the account.
- `window_minutes`: `int`, size of the rolling time window, in minutes (inclusive on both ends).

## Output
A **sorted, de-duplicated** `list[str]` of account names that hit the threshold at any point.

## Examples
```
events = [
  {"account":"mkt_dana","host":"h1","ts":0},
  {"account":"mkt_dana","host":"h2","ts":2},
  {"account":"mkt_dana","host":"h3","ts":4},
  {"account":"svc_backup","host":"h9","ts":0},
]
solve(events, threshold=3, window_minutes=5) -> ["mkt_dana"]
```

```
events = [
  {"account":"alice","host":"h1","ts":0},
  {"account":"alice","host":"h2","ts":100},
]
solve(events, threshold=2, window_minutes=5) -> []   # too far apart for the window
```

## Constraints
- Deterministic, offline, standard library only.
- A window `[t, t + window_minutes]` counts an event if its `ts` falls within that inclusive range;
  consider every event's `ts` as a possible window start.
- Repeated authentications to the SAME host do not add distinct-host count.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(events: list, threshold: int, window_minutes: int) -> list
```

### Input
- `events`: `list[dict]`, each `{"account": str, "host": str, "ts": int}`. Not necessarily sorted
  by `ts`. `ts` is minutes since the start of the log (non-negative int).
- `threshold`: `int`, minimum number of distinct hosts within a window to flag the account.
- `window_minutes`: `int`, inclusive window size in minutes.

### Output (`result`)
Sorted, de-duplicated `list[str]` of account names for which some window
`[start, start + window_minutes]` (with `start` equal to some event's `ts` for that account)
contains events touching `>= threshold` distinct hosts.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- The window bounds are inclusive: an event with `ts == start + window_minutes` counts.
- Repeated events to the same host within a window count once toward distinct-host count.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 5.2.c. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
from collections import defaultdict
from typing import Dict, List


def solve(events: List[Dict[str, object]], threshold: int, window_minutes: int) -> List[str]:
    """Flag accounts whose authentication events hit >= threshold distinct hosts
    within some rolling window of `window_minutes`."""
    by_account: Dict[str, List[Dict[str, object]]] = defaultdict(list)
    for ev in events:
        by_account[ev["account"]].append(ev)

    flagged = set()
    for account, evs in by_account.items():
        starts = sorted(set(e["ts"] for e in evs))
        for start in starts:
            window_hosts = set(
                e["host"] for e in evs if start <= e["ts"] <= start + window_minutes
            )
            if len(window_hosts) >= threshold:
                flagged.add(account)
                break

    return sorted(flagged)


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_lateral_movement_flagged():
    events = [
        {"account": "mkt_dana", "host": "h1", "ts": 0},
        {"account": "mkt_dana", "host": "h2", "ts": 2},
        {"account": "mkt_dana", "host": "h3", "ts": 4},
        {"account": "svc_backup", "host": "h9", "ts": 0},
    ]
    assert impl.solve(events, threshold=3, window_minutes=5) == ["mkt_dana"]


def test_too_far_apart_not_flagged():
    events = [
        {"account": "alice", "host": "h1", "ts": 0},
        {"account": "alice", "host": "h2", "ts": 100},
    ]
    assert impl.solve(events, threshold=2, window_minutes=5) == []


def test_same_host_repeated_not_counted_twice():
    events = [
        {"account": "bob", "host": "h1", "ts": 0},
        {"account": "bob", "host": "h1", "ts": 1},
        {"account": "bob", "host": "h1", "ts": 2},
    ]
    assert impl.solve(events, threshold=2, window_minutes=5) == []


def test_multiple_accounts_flagged_sorted():
    events = [
        {"account": "zed", "host": "h1", "ts": 0},
        {"account": "zed", "host": "h2", "ts": 1},
        {"account": "amy", "host": "h3", "ts": 0},
        {"account": "amy", "host": "h4", "ts": 1},
    ]
    assert impl.solve(events, threshold=2, window_minutes=5) == ["amy", "zed"]


# ---- hidden / edge cases -------------------------------------------------
def test_window_boundary_inclusive():
    events = [
        {"account": "u", "host": "h1", "ts": 0},
        {"account": "u", "host": "h2", "ts": 5},
    ]
    assert impl.solve(events, threshold=2, window_minutes=5) == ["u"]


def test_window_boundary_exclusive_just_outside():
    events = [
        {"account": "u", "host": "h1", "ts": 0},
        {"account": "u", "host": "h2", "ts": 6},
    ]
    assert impl.solve(events, threshold=2, window_minutes=5) == []


def test_unsorted_input_events():
    events = [
        {"account": "u", "host": "h3", "ts": 4},
        {"account": "u", "host": "h1", "ts": 0},
        {"account": "u", "host": "h2", "ts": 2},
    ]
    assert impl.solve(events, threshold=3, window_minutes=5) == ["u"]


def test_empty_events():
    assert impl.solve([], threshold=1, window_minutes=5) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
